# Clase 10 · Caso Panorama de la vivienda (1) — práctica guiada
**Objetivo:** reproducir la metodología del profesor: un **escenario básico** (mediana, sin quitar outliers) y medir cómo cambia **una sola métrica** (MAE) al cambiar el preprocesamiento.

El profesor usó el dataset de Melbourne (objetivo `Price`). Aquí usamos su análogo del repo: `data/raw/viviendas_lima.csv` (objetivo `precio_usd`).

Orden: 1) mirar faltantes · 2) separar X/y · 3) dividir train/test **una vez** · 4) `score_dataset` · 5) escenario 1 a mano · 6) la regla `fit_transform` / `transform` · 7) quitar outliers solo del train · Tu turno.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.metrics import mean_absolute_error
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401  (activa IterativeImputer)
from sklearn.impute import IterativeImputer

# Busca la raíz del repo (la carpeta que contiene data/raw) para que el notebook corra desde cualquier sitio
raiz = Path.cwd()
while not (raiz / "data" / "raw").exists() and raiz != raiz.parent:
    raiz = raiz.parent
df = pd.read_csv(raiz / "data" / "raw" / "viviendas_lima.csv")
print(df.shape)
df.head()

## 1. Mirar los faltantes y los tipos
El profesor: *"como ustedes saben, hay valores nulos"*. Y hay columnas de texto (`object`) que **todavía no** vamos a usar: el encoding llega en otra clase.

In [ ]:
resumen = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "%nulos": (df.isna().mean() * 100).round(1),
})
resumen

## 2. Separar y (objetivo) y X (predictores numéricos)
- `y` = lo que queremos predecir: `precio_usd`.
- `X` = todo lo demás, **sin el objetivo** y **sin columnas de texto** (`select_dtypes(exclude="object")`).
- 📌 Extra: también quitamos `id`. Es un número, pero no dice nada del precio; si lo dejamos, el modelo podría aprender ruido.

In [ ]:
y = df["precio_usd"]
X = df.drop(columns=["precio_usd", "id"]).select_dtypes(exclude="object")
print("Columnas de X:", list(X.columns))
print("NaN por columna en X:")
print(X.isna().sum())

## 3. Dividir en train y test (UNA sola vez)
*"80/20 y random_state=0; por favor mantengan esta configuración"*. Esta división **no pertenece a ningún escenario**: todos los escenarios usan los mismos train y test, así la comparación es justa.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print("train:", X_train.shape, " test:", X_test.shape)

## 4. `score_dataset`: el modelo fijo
Un **RandomForestRegressor**: 50 árboles (`n_estimators=50`), cada uno con profundidad máxima 6 (`max_depth=6`, para no sobreajustar), semilla fija. Devuelve el **MAE** = error absoluto medio, en dólares.

In [ ]:
def score_dataset(X_train, X_valid, y_train, y_valid):
    modelo = RandomForestRegressor(n_estimators=50, max_depth=6, random_state=0)
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_valid)
    return mean_absolute_error(y_valid, pred)

### ¿Qué significa el MAE? (ejemplo de juguete)
Si los precios reales son 100, 200 y 300 mil y el modelo predice 110, 180 y 300 mil, los errores absolutos son 10, 20 y 0 → MAE = 10 mil. *"En promedio me equivoco por 10 mil dólares."* **Menor es mejor.**

In [ ]:
mean_absolute_error([100_000, 200_000, 300_000], [110_000, 180_000, 300_000])

## 5. Escenario 1 (básico): mediana, sin quitar outliers
Los tres pasos del profesor:
1. **Definir el imputador** → `SimpleImputer(strategy="median")`.
2. **Eliminar outliers** → en el básico se **omite**.
3. **Entrenar y medir** → `score_dataset`.

Fíjate: `fit_transform` en train, **solo** `transform` en test.

In [ ]:
# Paso 1: definir el imputador
my_imputer = SimpleImputer(strategy="median")

# Aprender las medianas del TRAIN y aplicarlas a train y a test
X_train_imp = pd.DataFrame(my_imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_imp = pd.DataFrame(my_imputer.transform(X_test), columns=X_test.columns, index=X_test.index)

# Paso 2: (escenario básico) no quitamos outliers

# Paso 3: entrenar y medir
error = score_dataset(X_train_imp, X_test_imp, y_train, y_test)
print(f"Escenario 1 · mediana · MAE = {error:,.0f} USD")

## 6. La regla clave: ¿qué aprendió el imputador?
`fit` = **aprender** (aquí, la mediana de cada columna). `transform` = **aplicar** lo aprendido. Las medianas que se usaron en el test salen **solo del train**.

In [ ]:
comparacion = pd.DataFrame({
    "mediana aprendida (imputador)": my_imputer.statistics_,
    "mediana del train": X_train.median().values,
    "mediana del test": X_test.median().values,
}, index=X_train.columns)
comparacion

### 📌 Extra · Data leakage en miniatura
Si hicieras `fit` con el test (o `fit_transform` en el test), el modelo "espiaría" información del examen. En la vida real, el test representa casas **que todavía no has visto**: no puedes conocer su mediana.

In [ ]:
train_mini = pd.DataFrame({"banos": [1.0, 2.0, 3.0, np.nan]})
test_mini = pd.DataFrame({"banos": [np.nan, 9.0, 9.0]})

bien = SimpleImputer(strategy="median").fit(train_mini)
print("Correcto -> el NaN del test se rellena con", bien.transform(test_mini)[0, 0], "(mediana del train)")

mal = SimpleImputer(strategy="median").fit(test_mini)   # ¡fit con el test!
print("Leakage  -> el NaN del test se rellena con", mal.transform(test_mini)[0, 0], "(mediana del test: información prohibida)")

## 7. 📌 Extra · Quitar outliers SOLO del train
La consigna: eliminar outliers con **máximo 10 % de contaminación**. El profesor pidió PyOD (kNN y otros tres algoritmos). Como PyOD no está instalado, usamos `IsolationForest` de scikit-learn (también existe en PyOD).

Reglas:
- Se detectan **después de imputar** (estos algoritmos no aceptan NaN).
- Se filtran `X_train` **e** `y_train` con la **misma máscara** (si no, las filas se desalinean).
- El **test no se toca**: el modelo tiene que funcionar también con casas raras.

In [ ]:
detector = IsolationForest(contamination=0.1, random_state=0)
pred = detector.fit_predict(X_train_imp)      # 1 = inlier, -1 = outlier (convención de scikit-learn)
mascara = pred == 1

X_train_lim = X_train_imp[mascara]
y_train_lim = y_train[mascara]
print("Filas de train antes:", len(X_train_imp), " después:", len(X_train_lim))
print("¿Índices alineados?", (X_train_lim.index == y_train_lim.index).all())
print("Filas de test (sin tocar):", len(X_test_imp))

error_sin_out = score_dataset(X_train_lim, X_test_imp, y_train_lim, y_test)
print(f"Mediana + sin outliers · MAE = {error_sin_out:,.0f} USD  (básico: {error:,.0f})")

## ✏️ Tu turno 1 — Escenario con KNNImputer
Repite los pasos de la sección 5 pero con `KNNImputer(n_neighbors=5)`. ¿El MAE sube o baja frente al básico?

💡 Recuerda: `fit_transform` en train, `transform` en test.

In [ ]:
# Tu código aquí

## ✏️ Tu turno 2 — IterativeImputer + quitar outliers
Arma el escenario `IterativeImputer(random_state=0)` **con** eliminación de outliers (IsolationForest, contamination=0.1). Después prueba otro detector, por ejemplo `LocalOutlierFactor(contamination=0.1)` de `sklearn.neighbors` (su `fit_predict` también devuelve 1 / -1).

In [ ]:
# Tu código aquí

## ✏️ Tu turno 3 — Tabla de 5 escenarios
El profesor pidió **5 escenarios**. Junta tus resultados en un DataFrame con columnas `escenario`, `mae`, `filas_train`, ordénalo por `mae` y responde:
- ¿Qué escenario ganó? ¿Por cuántos dólares frente al básico?
- ¿Quitar outliers ayudó siempre, o depende del imputador?
- ¿Es una diferencia grande o pequeña comparada con el MAE total?

(La función `evaluar_escenarios` de `ejercicio.py` automatiza justo esto.)

In [ ]:
resultados = [
    {"escenario": "1_basico_mediana", "mae": error, "filas_train": len(X_train_imp)},
    {"escenario": "mediana_sin_outliers", "mae": error_sin_out, "filas_train": len(X_train_lim)},
    # agrega aquí tus escenarios
]
pd.DataFrame(resultados).sort_values("mae").reset_index(drop=True)